# Assignment 4 - Personalized FAQ Knowledge Base

This notebook solves all six questions in separate cells.


In [7]:
## Q1. Build the personalized FAQ DataFrame

import pandas as pd

roll_number = "23012345"  # replace with your actual roll number

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

category_map = ["billing", "account", "general"]
digits = [int(d) for d in str(roll_number)]
last_two_digits = digits[-2:]

personalized_entries = []
for d in last_two_digits:
    category = category_map[d % 3]
    if category == "billing":
        item = {
            "question": f"How do I download my fee receipt for roll number {d}?",
            "answer": "Go to the student portal, open Fee History, and download the receipt for the selected semester.",
            "keywords": "receipt fee payment invoice download",
            "category": "billing",
        }
    elif category == "account":
        item = {
            "question": f"How do I update my registered mobile number for student account {d}?",
            "answer": "Open your profile, go to Account Settings, and update your mobile number after verification.",
            "keywords": "mobile number account profile update",
            "category": "account",
        }
    else:
        item = {
            "question": f"Where can I find the timetable for class {d}?",
            "answer": "The timetable is available in the student portal under Academics > Timetable.",
            "keywords": "timetable classes schedule portal",
            "category": "general",
        }
    personalized_entries.append(item)

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print(faq_df)


                                            question  \
0                             what is the annual fee   
1                              how to reset password   
2                        what are your working hours   
3                              how can i pay the fee   
4  How do I update my registered mobile number fo...   
5        Where can I find the timetable for class 5?   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Open your profile, go to Account Settings, and...   
5  The timetable is available in the student port...   

                               keywords category  
0                 fee cost price charge  billing  
1                  password reset login  account  
2                hours timing open time  general  
3         

In [8]:
## Q2. Score a query against the FAQ entries

def score_query(query, df):
    terms = set(str(query).lower().split())
    matches = []
    for _, row in df.iterrows():
        keywords = set(str(row["keywords"]).lower().split())
        score = len(terms & keywords)
        if score > 0:
            matches.append({
                "question": row["question"],
                "category": row["category"],
                "score": score,
                "answer": row["answer"],
            })
    return sorted(matches, key=lambda x: (-x["score"], x["question"]))

query = "fee payment"
print(score_query(query, faq_df))


[{'question': 'how can i pay the fee', 'category': 'billing', 'score': 2, 'answer': 'You can pay via UPI, card, or net banking.'}, {'question': 'what is the annual fee', 'category': 'billing', 'score': 1, 'answer': 'The annual fee is Rs 500.'}]


In [9]:
## Q3. Return all FAQs in a chosen category

def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][["question", "answer", "keywords", "category"]]

selected_category = faq_df.iloc[4]["category"]
print(f"Category used: {selected_category}")
print(same_category(selected_category, faq_df))


Category used: account
                                            question  \
1                              how to reset password   
4  How do I update my registered mobile number fo...   

                                              answer  \
1                   Go to Settings > Reset Password.   
4  Open your profile, go to Account Settings, and...   

                               keywords category  
1                  password reset login  account  
4  mobile number account profile update  account  


In [10]:
## Q4. Add a keyword to an entry and save the updated CSV

entry_index = int(input("Enter the index of the FAQ entry to update (0-5): "))
new_keyword = input("Enter the new keyword to add: ").strip()

faq_df.loc[entry_index, "keywords"] = faq_df.loc[entry_index, "keywords"] + " " + new_keyword

file_name = f"{roll_number}_faq_data.csv"
faq_df.to_csv(file_name, index=False)
print(f"Saved updated FAQ data to {file_name}")
print(faq_df)


Saved updated FAQ data to 23012345_faq_data.csv
                                            question  \
0                             what is the annual fee   
1                              how to reset password   
2                        what are your working hours   
3                              how can i pay the fee   
4  How do I update my registered mobile number fo...   
5        Where can I find the timetable for class 5?   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Open your profile, go to Account Settings, and...   
5  The timetable is available in the student port...   

                               keywords category  
0                 fee cost price charge  billing  
1                  password reset login  account  
2            

In [11]:
## Q5. Count FAQ entries per category

print(faq_df.groupby("category").size().reset_index(name="count"))


  category  count
0  account      2
1  billing      2
2  general      2


In [12]:
## Q6. Handle score ties by printing every equally good match

def score_query_tie_aware(query, df):
    terms = set(str(query).lower().split())
    matches = []
    for _, row in df.iterrows():
        keywords = set(str(row["keywords"]).lower().split())
        score = len(terms & keywords)
        if score > 0:
            matches.append({
                "question": row["question"],
                "category": row["category"],
                "score": score,
                "answer": row["answer"],
            })
    if not matches:
        print("No matching FAQ entries found.")
        return []

    best_score = max(item["score"] for item in matches)
    best_matches = [item for item in matches if item["score"] == best_score]

    print(f"Best score: {best_score}")
    for item in best_matches:
        print(item)
    return best_matches

print("Tie example for query 'fee':")
score_query_tie_aware("fee", faq_df)
print("\nNon-tie example for query 'mobile':")
score_query_tie_aware("mobile", faq_df)


Tie example for query 'fee':
Best score: 1
{'question': 'what is the annual fee', 'category': 'billing', 'score': 1, 'answer': 'The annual fee is Rs 500.'}
{'question': 'how can i pay the fee', 'category': 'billing', 'score': 1, 'answer': 'You can pay via UPI, card, or net banking.'}

Non-tie example for query 'mobile':
Best score: 1
{'question': 'How do I update my registered mobile number for student account 4?', 'category': 'account', 'score': 1, 'answer': 'Open your profile, go to Account Settings, and update your mobile number after verification.'}


[{'question': 'How do I update my registered mobile number for student account 4?',
  'category': 'account',
  'score': 1,
  'answer': 'Open your profile, go to Account Settings, and update your mobile number after verification.'}]